In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path("/scratch2/ij292/tnp-crps")
RESULT_DIR = repo_root / "results" / "synthetic_1d" / "latent_fork_eval_full"

metrics_path = RESULT_DIR / "metrics.csv"
print(metrics_path)
print(metrics_path.exists())

metrics = pd.read_csv(metrics_path)

metrics["coverage90_error"] = metrics["coverage_90"] - 0.90
metrics["coverage95_error"] = metrics["coverage_95"] - 0.95
metrics["ssr_error"] = metrics["spread_skill_ratio"] - 1.0

MODEL_ORDER = [
    "latent_fork_gaussian_tnp_nll",
    "latent_fork_dropout_crps_m4_p010",
    "latent_fork_stochln_crps_m4",
]

MODEL_LABELS = {
    "latent_fork_gaussian_tnp_nll": "Gaussian TNP NLL",
    "latent_fork_dropout_crps_m4_p010": "Dropout CRPS M=4 p=.10",
    "latent_fork_stochln_crps_m4": "StochLN CRPS M=4",
}

def add_display_labels(df):
    out = df.copy()
    out["Model"] = out["model_name"].map(MODEL_LABELS).fillna(out["model_name"])
    out["model_order"] = out["model_name"].map({m: i for i, m in enumerate(MODEL_ORDER)})
    return out.sort_values(["model_order"])

main = metrics.query(
    "eval_set == 'latent_fork' and region == 'all' and context_bucket == 'all'"
).copy()

main = add_display_labels(main)

cols = [
    "Model",
    "rmse_pooled",
    "crps",
    "spread_skill_ratio",
    "coverage_90",
    "coverage_95",
    "width_90",
    "width_95",
]

if "energy_score" in main.columns:
    cols.insert(3, "energy_score")

table = main[cols].rename(
    columns={
        "rmse_pooled": "RMSE ↓",
        "crps": "CRPS ↓",
        "energy_score": "Energy ↓",
        "spread_skill_ratio": "SSR ≈ 1",
        "coverage_90": "Cov90",
        "coverage_95": "Cov95",
        "width_90": "Width90 ↓",
        "width_95": "Width95 ↓",
    }
)

for col in table.columns:
    if col != "Model":
        table[col] = table[col].round(3)

table

/scratch2/ij292/tnp-crps/results/synthetic_1d/latent_fork_eval_full/metrics.csv
True


/tmp/ipykernel_1453089/1563468620.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  metrics["coverage90_error"] = metrics["coverage_90"] - 0.90
/tmp/ipykernel_1453089/1563468620.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  metrics["coverage95_error"] = metrics["coverage_95"] - 0.95
/tmp/ipykernel_1453089/1563468620.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat

,Model,RMSE ↓,CRPS ↓,Energy ↓,SSR ≈ 1,Cov90,Cov95,Width90 ↓,Width95 ↓
9,Gaussian TNP NLL,1.513,0.798,11.800,1.001,0.877,0.932,4.381,5.127
0,Dropout CRPS M=4 p=.10,1.518,0.794,11.717,1.004,0.878,0.926,4.306,4.855
18,StochLN CRPS M=4,1.516,0.793,11.988,1.014,0.879,0.929,4.341,4.926


In [3]:
# save as png

def save_table_png(df, title, output_path, font_size=10, scale_y=1.35):
    fig_width = max(9, 1.25 * len(df.columns))
    fig_height = max(2.2, 0.45 * len(df) + 1.2)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    ax.set_title(title, fontsize=14, pad=14)

    tab = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
    )
    tab.auto_set_font_size(False)
    tab.set_fontsize(font_size)
    tab.scale(1.0, scale_y)

    for (row, col), cell in tab.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#eeeeee")

    fig.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=250, bbox_inches="tight")
    fig.savefig(output_path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return output_path

save_table_png(
    table,
    "Latent fork results",
    RESULT_DIR / "latent_fork_results_table.png",
)

PosixPath('/scratch2/ij292/tnp-crps/results/synthetic_1d/latent_fork_eval_full/latent_fork_results_table.png')